# MetaBioTarget AI: Autonomous Multi-Scale Drug Discovery Platform

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/metabotarget-ai/metabotarget-platform/blob/main/notebooks/MetaBioTarget_QuickStart.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**MetaBioTarget AI** is a modular computational platform that connects real-world public databases (UniProt, RCSB PDB, Human Protein Atlas, Open Targets Platform) with physics-grounded ODE solvers, context-aware meta-learning (MAML/FiLM), and population pharmacokinetics to discover, prioritize, and stress-test target-indication pairings.

### Interactive Workflow in this Notebook:
1. **Live Data Ingestion**: Pulls sequence, structures, and whole-body tissue expression dynamically.
2. **Multi-Scale Biology (Layer 1)**: Computes Molecular, Cellular ODE, Tissue B_max, and Organ PBPK scales.
3. **Context-Aware Meta-Learning**: Adapts predictions across species, microenvironment, dose, and time.
4. **Dynamic Indication Classifier**: Scores clinical indications from Open Targets Platform GraphQL.
5. **Virtual Population Trial**: Simulates N=150 virtual human patient PK/PD exposure.
6. **2-Cycle Stress-Testing Convergence**: Automatically closes translational discrepancies.

In [ ]:
# Cell 1: Environment Setup & Package Dependencies
import sys
import subprocess

print("[1/2] Installing required core scientific dependencies...")
!pip install -q numpy scipy requests matplotlib pydantic

# Clone or configure repository path
import os
if not os.path.exists("src") and not os.path.exists("metabotarget"):
    print("[2/2] Running in standalone Colab environment. Importing package...")
    # When uploaded to GitHub, users can clone or pip-install directly:
    # !git clone https://github.com/your-username/metabotarget-mvp.git && cd metabotarget-mvp
print("Environment initialized successfully!")

In [ ]:
# Cell 2: Target Selection & Live Multi-Omic Ingestion
# Choose any target by UniProt ID or Gene Symbol (e.g. 'P43220' / 'GLP1R', 'P02786' / 'TFRC', 'P01375' / 'TNF')
TARGET_IDENTIFIER = "P43220"  # Human GLP1R

from src.ingestion.live_connectors import LiveTargetAggregator

print(f"Fetching live multi-scale target data for {TARGET_IDENTIFIER}...")
aggregator = LiveTargetAggregator()
target_profile = aggregator.fetch_target_profile(TARGET_IDENTIFIER, max_diseases=20)

protein = target_profile["protein"]
print(f"Target Resolved: {protein['gene_symbol']} - {protein['protein_name']}")
print(f"UniProt Accession: {protein['accession']} | Length: {protein['sequence_length_aa']} AA | MW: {protein['molecular_weight_da']:.1f} Da")
print(f"Ensembl Gene ID: {target_profile.get('ensembl_id')}")
print(f"PDB Structures Available: {protein.get('pdb_ids', [])[:5]}")
print(f"Associated Clinical Diseases in Open Targets: {len(target_profile['associated_indications'])}")

In [ ]:
# Cell 3: Layer 1 Component 1 (The 4 Multi-Scale Biology Levels)
from src.meta_learning.component1_engine import Component1DynamicEngine

print("Computing all 4 Multi-Scale Biology levels from live data...")
c1_engine = Component1DynamicEngine()
c1_data = c1_engine.generate_full_component1_profile(TARGET_IDENTIFIER)

print("\n--- LEVEL 1: MOLECULAR SCALE ---")
mol = c1_data["molecular_scale"]
print(f"Sequence Length: {mol['sequence_length_aa']} aa | MW: {mol['molecular_weight_da']} Da")
print(f"Net Charge (pH 7.4): {mol['net_charge_ph74']} e | Isoelectric Point (pI): {mol['isoelectric_point_pI']}")
print(f"Active Cavity Volume: {mol['active_cavity_volume_A3']} A3 | Surface Potential: {mol['surface_potential_mV']} mV")

print("\n--- LEVEL 2: CELLULAR / PATHWAY ODE SCALE ---")
cell = c1_data["cellular_pathway_scale"]
print(f"Intracellular Signaling Peak: {cell['camp_peak_pmol_per_1e6_cells']} pmol (Radau Stiff Solver)")
print(f"Receptor Internalization (5 min): {cell['receptor_internalization_5min_pct']}% | Transcytosis Yield: {cell['transcytosis_yield_1h_pct']}%")

print("\n--- LEVEL 3: TISSUE SCALE (HUMAN PROTEIN ATLAS) ---")
tiss = c1_data["tissue_scale"]
print("Top Ranked Tissues by Functional Burden (B_max * Coupling):")
for t in tiss["top_ranked_tissues_by_functional_burden"][:4]:
    print(f"  * {t['tissue']}: Expression = {t['ntpm_expression']} nTPM | B_max = {t['bmax_fmol_mg']} fmol/mg | Score = {t['functional_burden_score']}")

print("\n--- LEVEL 4: ORGAN PBPK SCALE ---")
org = c1_data["organ_scale"]
print(f"Volume of Distribution: {org['steady_state_volume_of_distribution_L']} L | Total Clearance: {org['total_systemic_clearance_L_hr']} L/hr")
print(f"Elimination Half-Life: {org['predicted_circulating_half_life_days']} days | Brain PS Product: {org['brain_permeability_surface_area_product_uL_min']} uL/min")

In [ ]:
# Cell 4: Multi-Agent Governance & Dynamic Indication Classification (Components 2, 3, and 4)
from src.agents.platform_orchestrator import PlatformMultiAgentOrchestrator

orchestrator = PlatformMultiAgentOrchestrator()
print(f"Executing Layer 1 Pipeline under {orchestrator.agent_name}...")

pipeline_out = orchestrator.execute_layer1_pipeline_with_enforcement(
    target_symbol=c1_data["gene_symbol"],
    component_1_data=c1_data,
    probability_threshold=0.45,
)

print(f"\n>>> Multi-Agent Audit: {pipeline_out['non_placeholder_enforcement']['decision']}")
print(f">>> Component 2 FiLM Context Modulation Applied (7 Factors: species, cell, donor, genotype, microenv, dose, time)")
print(f">>> Component 3 Causal Cross-Species Alignment: {pipeline_out['component_3']['causal_cross_species_alignment_cosine']} ({pipeline_out['component_3']['conservation_status']})")
print(f">>> Component 3 Uncertainty: Total = {pipeline_out['component_3']['total_uncertainty']} | 95% CI = {pipeline_out['component_3']['confidence_interval_95']}")

c4 = pipeline_out["component_4"]
print(f"\n>>> Component 4 Dynamic Indication Classifier ({c4['model_architecture']}):")
print(f"    Total Diseases Evaluated from Live Open Targets: {c4['indication_universe_size']} | Classified: {c4['classified_indications_count']}")
print("\nTop Classified Clinical Indications:")
for ind in c4["ranked_indications"][:6]:
    print(f"  [{ind['rank']}] Probability = {ind['probability_score']:.4f} | {ind['indication']} (ID: {ind['id']}) - Primary Driver: {ind['primary_driving_feature']}")

In [ ]:
# Cell 5: Virtual Human Clinical Trial (N=150 Virtual Cohort Monte Carlo Simulation)
from src.translational.simulation.population_engine import PopulationPKSimulator
from src.translational.pkpd.dose_optimizer import DoseRegimenOptimizer
import matplotlib.pyplot as plt

simulator = PopulationPKSimulator(seed=42)
cohort = simulator.sample_cohort(n_cohort=150, base_organ_params=c1_data["organ_scale"])

print(f"Sampled Virtual Human Cohort (N={len(cohort)}) with ICRP physiological distributions.")
pk_res = simulator.simulate_cohort_exposure(cohort=cohort, dose_mg=10.0, dosing_interval_h=168.0, ec90_nM=2.5)

print(f"Cohort Mean Cmax: {pk_res['cmax_mean_nM']:.2f} nM | AUC: {pk_res['auc_mean_nM_h']:.1f} nM*h")
print(f"Cohort Trough Concentration (P10 - P90): {pk_res['trough_p10_nM']} - {pk_res['trough_p90_nM']} nM")
print(f"Population Responder Rate (Trough > EC90): {pk_res['population_responder_rate_pct']}%")

# Dose Optimization
optimizer = DoseRegimenOptimizer(simulator)
opt_res = optimizer.optimize_dose(cohort, target_responder_rate_pct=80.0)
best_dose = opt_res["optimal_regimen"]
print(f"\n>>> Recommended Optimal Regimen: {best_dose['dose_mg']} mg {best_dose['interval_label']} (Responder Rate: {best_dose['responder_rate_pct']}%, Cmax: {best_dose['cmax_mean_nM']:.1f} nM)")

In [ ]:
# Cell 6: Phase 12 - Two-Cycle Stress-Testing Discovery Engine & Final Visualizations
from src.orchestration.closed_loop_engine import TwoCycleStressTestEngine

print("Running Autonomous Two-Cycle Stress-Testing Engine...")
two_cycle_engine = TwoCycleStressTestEngine()
discovery_results = two_cycle_engine.run_two_cycle_discovery(TARGET_IDENTIFIER, n_virtual_cohort=100)

print(f"\n>>> Discovery Status: {discovery_results['discovery_status']}")
print(f">>> Total Cycles: {discovery_results['total_cycles_executed']}")
print(f">>> Summary: {discovery_results['convergence_summary']}")

# Plot Ranked Indications
indications = [ind['indication'][:25] for ind in discovery_results['final_indications'][:6]]
probs = [ind['probability_score'] for ind in discovery_results['final_indications'][:6]]

plt.figure(figsize=(10, 4))
bars = plt.barh(indications[::-1], probs[::-1], color="#2b5c8f")
plt.xlabel("Classification Probability P(Indication | Target, Context)")
plt.title(f"MetaBioTarget Discovery: Top Ranked Indications for {c1_data['gene_symbol']}")
plt.xlim(0, 1.05)
for bar in bars:
    plt.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()
print("Discovery Workflow Complete!")